# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. This dataset contains ordered logistic regression outputs for adoption predictors of indigenous and modern knowledge in rangeland management interventions.

### Dataset Source
The dataset Croissant schema is provided via this URL: https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and prepare for exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}\nPublished: {metadata.datePublished}\nIdentifier: {metadata.identifier}\n")

## 2. Data Overview
Review the available record sets and their fields, using the `@id` for referencing. This helps in selecting data for further analysis.

**Note**: In the Croissant schema, `recordSet` entities define logical tables or record collections. Fields and columns within a record set also have their own `@id`s.

In [ ]:
# List all available record sets (by their @id) and print their fields
from collections import defaultdict

record_sets = list(dataset.record_sets)
if len(record_sets) == 0:
    # This dataset defines record sets only after accessing records; let's enumerate record sets via ds.record_sets
    print('No record sets explicitly declared in Croissant schema.')
else:
    print("Available record sets and fields:\n")
    for record_set in record_sets:
        print(f"Record set @id: {record_set['@id']}")
        print(f"  Name: {record_set.get('name')}")
        print(f"  Description: {record_set.get('description')}")
        field_ids = [f.get('@id') for f in record_set.get('field', [])]
        print(f"  Field @ids: {field_ids}")
        print()

# Alternatively, try listing all record sets as recognized by mlcroissant
if hasattr(dataset, 'record_sets'):
    record_sets = list(dataset.record_sets)
    if len(record_sets):
        print('\nFound record sets via dataset.record_sets (by @id):')
        for rs in record_sets:
            print(f" - {rs['@id']}")

Let's attempt to preview records of available record sets. If the dataset only provides one primary record set (typical for regression result tables), we'll load it directly.

For all data operations, we will reference record sets and fields by their `@id` to ensure formal disambiguation.

In [ ]:
# Automatically list all record set @ids recognized by dataset.records()
# Most datasets have record set ids, e.g. cr:OrderedLogitResults
record_set_ids = mlc.utils.record_set_ids(dataset)
if len(record_set_ids) == 0:
    raise ValueError('No record sets found in this Croissant schema.')
print("Record set @ids in the dataset:")
for rsid in record_set_ids:
    print(f"- {rsid}")

# We'll display a few records for each record set
for rsid in record_set_ids:
    print(f"\nSample records for record set: {rsid}")
    for i, rec in enumerate(dataset.records(record_set=rsid)):
        print(rec)
        if i>=2:  # Print up to 3 records
            break

## 3. Data Extraction

Now we'll load all record sets into Pandas DataFrames. We'll use each record set's `@id` as the dictionary key. This prepares our data for analysis and EDA.

**Note:** Field and column names will match their `@id`s as specified in the Croissant schema.

In [ ]:
# Extract all record sets to DataFrames, using their @id
dataframes = {}

for rsid in record_set_ids:
    df = pd.DataFrame(list(dataset.records(record_set=rsid)))
    dataframes[rsid] = df

# Show columns of the first record set and preview data
selected_record_set_id = record_set_ids[0]
print(f"Columns for record set '@id': {selected_record_set_id}")
print(dataframes[selected_record_set_id].columns.tolist())
dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

We will:
- Filter records using a numeric field referenced by its `@id` (e.g., coefficients/p-values/log-likelihood).
- Create a normalized version of this numeric field.
- Optionally, group by a categorical field (by `@id`).

> **Tip:** Use the DataFrame's columns (which are field `@id`s) to select the best numeric/categorical fields for EDA.

If running this notebook interactively, inspect `dataframes[selected_record_set_id].columns` to identify available fields.

In [ ]:
# Choose the first available numeric field for demonstration

df = dataframes[selected_record_set_id]

# Attempt to auto-select a numeric field
possible_numeric_types = (int, float)

numeric_field_id = None
for col in df.columns:
    sample_vals = df[col].dropna().head(10)
    if all(isinstance(v, possible_numeric_types) or pd.api.types.is_number(v) for v in sample_vals):
        numeric_field_id = col
        break

if numeric_field_id is None:
    raise ValueError('No numeric field found for EDA in record set.')
print(f"Numeric field chosen (by @id): {numeric_field_id}")

# Filtering by an arbitrary threshold
threshold = df[numeric_field_id].mean()  # Use mean as demonstration threshold
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.4f}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / 
    filtered_df[numeric_field_id].std()
)
print(f"\nNormalized field '{numeric_field_id}_normalized':")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt group-by for a categorical/grouping field
group_field = None
for col in df.columns:
    if col != numeric_field_id and df[col].nunique() < 20 and df[col].dtype==object:
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"\nMean {numeric_field_id} grouped by {group_field} (by @id):")
    print(grouped_df)

## 5. Visualization

Visualize the distribution of the selected numeric field and, when possible, compare means by group.
This helps in understanding patterns, spread, and outliers within the logistic regression results.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the (normalized) numeric field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=20, kde=True, color="teal")
plt.title(f"Distribution of Normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id}_normalized")
plt.tight_layout()
plt.show()

# If grouping is possible, show a barplot of means
if group_field:
    plt.figure(figsize=(8, 4))
    sns.barplot(
        data=filtered_df, x=group_field, y=numeric_field_id,
        ci=None, estimator=pd.Series.mean, palette="muted"
    )
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

Using the `mlcroissant` library, we have:
- Loaded the FAIR² regression results dataset from a Croissant schema.
- Explored provided record sets, fields, and loaded tabular outputs by referencing their `@id`s.
- Conducted introductory data filtering, normalization, and group-based aggregation.
- Visualized distributions and grouped averages to inform further statistical/data science analyses.

This notebook can now be adapted to richer analyses and reporting, thanks to the formal structure provided by the Croissant standard. For further exploration, consider feature selection, statistical summaries, or export to additional downstream tools.